# PathSnap: 03. HMM & Viterbi Sequential Route Reconstruction
### Distributed Map-Matching & Route Reconstruction

This notebook conducts experimental validation of sequential route reconstruction using Hidden Markov Models (HMM) and Viterbi decoding:
1. **Hidden Markov Model Formulation**:
   - **Hidden states**: Candidate road network arcs
   - **Observations**: Observed GPS sequence
   - **Emission probabilities**: Combination of Gaussian orthogonal distance and classifier confidence
   - **Transition probabilities**: Topological connectivity and route length consistency
2. **Log-space Viterbi Trellis Algorithm**:
   - Exact MAP sequence decoding
   - Robust handling of GPS gaps and disconnected components
3. **Ablation & Comparative Analysis**:
   - Classifier-Only (Pointwise greedy matching) vs.
   - Classifier + HMM Viterbi Sequential Matching
4. **Route Continuity & Observation Top-1 Accuracy evaluation**
5. **Interactive Folium map generation**


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path(".").resolve().parent))

from src.data.loader import TripLoader
from src.features.candidates import CandidateGenerator
from src.features.engineering import FeatureExtractor
from src.models.train import load_model
from src.models.evaluate import compute_route_continuity
from src.map_matching.hmm import MapMatchingHMM
from src.map_matching.viterbi import viterbi_decode
from src.visualization.maps import create_folium_map

loader = TripLoader("../data")
cg = CandidateGenerator(search_radius_m=120.0, top_k=6)
fe = FeatureExtractor()
hmm = MapMatchingHMM(sigma_z=20.0, beta=35.0)

# Load trained classifier pipeline
pipeline = load_model("../artifacts/model.joblib")


## 1. Run Map-Matching on Student-Test Trip


In [ ]:
sample_trip_id = loader.list_trips("student_test")[0]
trip = loader.load_trip(sample_trip_id, split="student_test")

track = trip["track"][:200]
arcs = trip["arcs"]
nodes = trip["nodes"]
route = trip["route"]

# Step 1: Candidate Generation
candidates_by_point = cg.generate_candidates(track, arcs, nodes)

# Step 2: Classifier Probability Scoring
df_feat = fe.extract_features_for_trip(track, arcs, nodes, candidates_by_point)
X = df_feat[fe.FEATURE_NAMES].values
probs = pipeline.predict_proba(X)[:, 1]
df_feat["prob"] = probs

probs_by_point = [
    df_feat[df_feat["point_idx"] == pt_idx]["prob"].tolist()
    for pt_idx in range(len(track))
]

# Step 3: Classifier-Only Greedy Matching
greedy_matched_arcs = [
    cands[np.argmax(p)]["arc_id"] if len(cands) > 0 else -1
    for cands, p in zip(candidates_by_point, probs_by_point)
]

# Step 4: Classifier + HMM Viterbi Sequential Decoding
viterbi_matched_arcs, projected_coords = viterbi_decode(
    track, candidates_by_point, arcs, nodes, hmm=hmm, classifier_probs_by_point=probs_by_point
)


## 2. Quantitative Comparison: Classifier-Only vs. HMM Decoding


In [ ]:
route_set = set(route) if route is not None else set()

# Top-1 observation accuracy
acc_greedy = sum(1 for a in greedy_matched_arcs if a in route_set) / len(track)
acc_viterbi = sum(1 for a in viterbi_matched_arcs if a in route_set) / len(track)

# Route continuity
cont_greedy = compute_route_continuity(greedy_matched_arcs, arcs)
cont_viterbi = compute_route_continuity(viterbi_matched_arcs, arcs)

comparison_summary = pd.DataFrame([
    {
        "Method": "Classifier-Only (Greedy)",
        "Top-1 Accuracy": f"{acc_greedy * 100:.2f}%",
        "Route Continuity": f"{cont_greedy * 100:.2f}%",
        "Sequence Consistency": "Fragmented hops"
    },
    {
        "Method": "Classifier + HMM Viterbi",
        "Top-1 Accuracy": f"{acc_viterbi * 100:.2f}%",
        "Route Continuity": f"{cont_viterbi * 100:.2f}%",
        "Sequence Consistency": "Topologically continuous"
    }
])
comparison_summary


## 3. Interactive Folium Visual Map Comparison


In [ ]:
# Extract coordinates for ground truth route
ref_coords = []
if route is not None:
    for arc_id in route:
        if arc_id < len(arcs):
            u, v = arcs[arc_id, 0], arcs[arc_id, 1]
            ref_coords.append([float(nodes[u, 0]), float(nodes[u, 1])])
            ref_coords.append([float(nodes[v, 0]), float(nodes[v, 1])])

fmap = create_folium_map(
    track=track,
    predicted_coords=projected_coords,
    ground_truth_coords=ref_coords,
    trip_id=sample_trip_id,
)
fmap.save("../results/plots/hmm_reconstruction_map.html")
print("Saved map visualization to results/plots/hmm_reconstruction_map.html")
